In [5]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

df_ca = pd.read_csv('california_preprocessed.csv')
df_ar = pd.read_csv('arkansas_preprocessed.csv')

# Features
X_ca = df_ca.drop(columns=['cropland', '.geo']).values
X_ar = df_ar.drop(columns=['cropland', '.geo']).values

# Remapping des labels
mapping_ca = {69: 0, 3: 1, 36: 2, 75: 3, 76: 4}
mapping_ar = {1: 0, 2: 1, 3: 2, 5: 3}

y_ca = df_ca['cropland'].map(mapping_ca).fillna(5).astype(int).values
y_ar = df_ar['cropland'].map(mapping_ar).fillna(4).astype(int).values

# Split 70/15/15 California
X_ca_train, X_ca_temp, y_ca_train, y_ca_temp = train_test_split(
    X_ca, y_ca, test_size=0.30, random_state=42, stratify=y_ca
)
X_ca_val, X_ca_test, y_ca_val, y_ca_test = train_test_split(
    X_ca_temp, y_ca_temp, test_size=0.50, random_state=42, stratify=y_ca_temp
)

# Split 70/15/15 Arkansas
X_ar_train, X_ar_temp, y_ar_train, y_ar_temp = train_test_split(
    X_ar, y_ar, test_size=0.30, random_state=42, stratify=y_ar
)
X_ar_val, X_ar_test, y_ar_val, y_ar_test = train_test_split(
    X_ar_temp, y_ar_temp, test_size=0.50, random_state=42, stratify=y_ar_temp
)

print(f"California — Train: {X_ca_train.shape}, Val: {X_ca_val.shape}, Test: {X_ca_test.shape}")
print(f"Arkansas   — Train: {X_ar_train.shape}, Val: {X_ar_val.shape}, Test: {X_ar_test.shape}")

# DataLoaders
batch_size = 64

train_loader_ca = DataLoader(TensorDataset(torch.FloatTensor(X_ca_train), torch.LongTensor(y_ca_train)), batch_size=batch_size, shuffle=True)
val_loader_ca   = DataLoader(TensorDataset(torch.FloatTensor(X_ca_val),   torch.LongTensor(y_ca_val)),   batch_size=batch_size, shuffle=False)
test_loader_ca  = DataLoader(TensorDataset(torch.FloatTensor(X_ca_test),  torch.LongTensor(y_ca_test)),  batch_size=batch_size, shuffle=False)

train_loader_ar = DataLoader(TensorDataset(torch.FloatTensor(X_ar_train), torch.LongTensor(y_ar_train)), batch_size=batch_size, shuffle=True)
val_loader_ar   = DataLoader(TensorDataset(torch.FloatTensor(X_ar_val),   torch.LongTensor(y_ar_val)),   batch_size=batch_size, shuffle=False)
test_loader_ar  = DataLoader(TensorDataset(torch.FloatTensor(X_ar_test),  torch.LongTensor(y_ar_test)),  batch_size=batch_size, shuffle=False)

print("DataLoaders prêts")

California — Train: (21288, 396), Val: (4562, 396), Test: (4562, 396)
Arkansas   — Train: (25637, 396), Val: (5494, 396), Test: (5494, 396)
DataLoaders prêts


In [ ]:
import torch
import torch.nn as nn

class MCTNet(nn.Module):
    def __init__(self, input_size=396, num_classes=6, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=32, out_channels=d_model, kernel_size=3, padding=1),
            nn.ReLU()
        )
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            batch_first=True,
            dim_feedforward=128
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        x = self.transformer(x)
        x = x.mean(dim=1)
        x = self.classifier(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

def train_model(model, train_loader, val_loader, num_epochs=30, lr=0.001):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_val_acc = 0
    best_model_state = None
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            predicted = outputs.argmax(dim=1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
        
        train_acc = 100 * correct / total
        
        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                predicted = outputs.argmax(dim=1)
                correct += (predicted == y_batch).sum().item()
                total += y_batch.size(0)
        
        val_acc = 100 * correct / total
        
        # Sauvegarder le meilleur modèle
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
        
        print(f"Epoch {epoch+1}/{num_epochs} — Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
    
    print(f"\n Meilleure Val Acc : {best_val_acc:.2f}%")
    model.load_state_dict(best_model_state)
    return model

# Entraînement
print("=== California ===")
model_ca = MCTNet(input_size=396, num_classes=6)
model_ca = train_model(model_ca, train_loader_ca, val_loader_ca)

print("\n=== Arkansas ===")
model_ar = MCTNet(input_size=396, num_classes=5)
model_ar = train_model(model_ar, train_loader_ar, val_loader_ar)

torch.save(model_ca.state_dict(), 'model_california.pth')
torch.save(model_ar.state_dict(), 'model_arkansas.pth')
print("Modèles sauvegardés")

Device : cuda
=== California ===
Epoch 1/30 — Loss: 1.2887 | Train Acc: 49.84% | Val Acc: 56.90%
Epoch 2/30 — Loss: 1.0281 | Train Acc: 62.10% | Val Acc: 66.53%
Epoch 3/30 — Loss: 0.8805 | Train Acc: 68.49% | Val Acc: 69.90%
Epoch 4/30 — Loss: 0.7997 | Train Acc: 71.91% | Val Acc: 73.50%
Epoch 5/30 — Loss: 0.7269 | Train Acc: 74.66% | Val Acc: 76.13%
Epoch 6/30 — Loss: 0.6886 | Train Acc: 76.04% | Val Acc: 77.58%
Epoch 7/30 — Loss: 0.6612 | Train Acc: 76.91% | Val Acc: 78.21%
Epoch 8/30 — Loss: 0.6299 | Train Acc: 78.07% | Val Acc: 81.35%
Epoch 9/30 — Loss: 0.5919 | Train Acc: 79.34% | Val Acc: 81.41%
Epoch 10/30 — Loss: 0.5798 | Train Acc: 80.09% | Val Acc: 80.89%
Epoch 11/30 — Loss: 0.5463 | Train Acc: 81.33% | Val Acc: 81.96%
Epoch 12/30 — Loss: 0.5432 | Train Acc: 81.44% | Val Acc: 82.88%
Epoch 13/30 — Loss: 0.5200 | Train Acc: 82.39% | Val Acc: 83.12%
Epoch 14/30 — Loss: 0.5007 | Train Acc: 82.97% | Val Acc: 80.97%
Epoch 15/30 — Loss: 0.4900 | Train Acc: 83.42% | Val Acc: 83.89%
E

In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3060
